## Rebuilding a Linear Congruential Generator in Python
- Testing uniformity and short-range correlation in pseudo-random number generation

This project rebuilds a pseudo-random number generator from an older ROOT/C++ script using Python. The generator is a linear congruential generator (LCG), which produces a sequence of integers using a recurrence relation, then rescales them into values between 0 and 1. I evaluate the sequence using a histogram for uniformity and a lag plot to inspect short-range correlation between values.

## Mathematical Formulation

This project uses a linear congruential generator (LCG) to produce pseudo-random numbers.

The recurrence relation is

$$
x_{n+1} = (a x_n + c) \bmod M
$$

where:
- $a$ is the multiplier
- $c$ is the increment
- $M$ is the modulus
- $x_0$ is the initial seed

To convert the generated integers into values on the interval $[0,1]$, the sequence is normalized as

$$
u_n = \frac{x_n}{M - 1}
$$

For this implementation:
- $a = 16807$
- $c = 0$
- $M = 2147483647$
- $x_0 = 3$

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt


In [ ]:
# Parameters 
a = 16807
c = 0
M = 2147483647
seed = 3
N = 100
lag = 4


In [ ]:
# Generator function cell
def generate_lcg(a, c, M, seed, N):
    values = []
    x = seed 

    for _ in range(N):
        x = (a * x + c) % M
        u = x / (M - 1)
        values.append(u)

    return np.array(values)

In [ ]:
# Generate sequence cell
values = generate_lcg(a, c, M, seed, N)

print("First 10 generated values: ")
print(values[:10])

A histogram helps check whether the generated values are spread approximately uniformly across the interval from 0 to 1.

In [ ]:
# Histogram
plt.figure(figsize=(7,4))
plt.hist(values, bins = 20, edgecolor = 'black')
plt.xlabel("Generated value")
plt.ylabel("Count")
plt.title("Histogram of LCG Values")
plt.savefig("histogram.png", dpi=300, bbox_inches="tight")
plt.show()


# Lag plot 
A lag plot compares each value with a value generated several steps earlier. If the genreator has obvious short-range structure, patterns may appear in the scatter plot. Here I compare $u_i$ with $u_{i-4}$.

In [ ]:
# Lag Plot
x_lag = values[lag:]
y_lag = values[:-lag]

plt.figure(figsize=(6,6))
plt.scatter(x_lag, y_lag)
plt.xlabel("u_n")
plt.ylabel("u_{n-1}")
plt.title("Lag Plot of LCG Values")
plt.savefig("lag_plot.png", dpi=300, bbox_inches="tight")
plt.show()

# Interpretation 

The histogram gives a basic visual check of whether the sequence appears roughly uniform. The lag plot gives a simple check for visible correlation between nearby values in the sequence. Together, these plots provide diagnostics of whether the generator behaves like a reasonable source of pseudo-random numbers at a basic level. 

# Reflection / Improvement

Compared with the original ROOT version, this Python rebuild improves readability by separating parameters, generation logic, and visualization into clear notebook sections. I then extended the project by using the generated values in a Monte Carlo estimate of $\pi$, showing how a custom pseudo-random number generator can support a real simulation-based calculation.

A natural future extension would be to compare this generator with NumPy’s default random number generator or to apply the same sampling method to another estimation problem.

## Monte Carlo Application: Estimating $\pi$

To extend the project beyond testing the generator itself, I use the generated values in a Monte Carlo estimate of $\pi$.

Random points $(x,y)$ are sampled in the unit square. A point lies inside the quarter circle if

$$
x^2 + y^2 \le 1
$$

Since the area of the quarter circle is $\pi/4$, the estimate is

$$
\pi \approx 4 \cdot \frac{\text{points inside the quarter circle}}{\text{total number of points}}
$$

In [ ]:
N_mc = 10000
mc_values = generate_lcg(a, c, M, seed, 2* N_mc)

x_points = mc_values[0::2]
y_points = mc_values[1::2]

inside = (x_points**2 + y_points**2) <= 1

pi_estimate = 4 * np.sum(inside) / len(inside)

print(f"Estimated pi: {pi_estimate}")
print(f"True pi: {np.pi}")
print(f"Absolute error: {abs(pi_estimate - np.pi)}")
print(f"Number of points used: {len(inside)}")

In [ ]:
plt.figure(figsize= (6,6))
plt.scatter(x_points[inside], y_points[inside], s=5, alpha=0.5, label="Inside circle")
plt.scatter(x_points[~inside], y_points[~inside], s=5, alpha=0.5, label="Outside circle")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Monte Carlo Estimate of Pi")
plt.legend()
plt.show()

## Interpretation of the Monte Carlo Estimate

After testing the pseudo-random number generator with a histogram and lag plot, I used the generated values in a Monte Carlo estimate of $\pi$. By sampling points in the unit square and checking whether they fall inside the quarter circle, I estimated $\pi$ from the ratio of accepted points to total points.

As the number of sampled points increases, the estimate generally becomes more stable. This extends the project from a random number generator diagnostic into a true simulation-based estimation problem.

## Conclusion

This project rebuilt a linear congruential generator from an older ROOT/C++ script in Python. I first evaluated the generated sequence using a histogram and a lag plot to check whether the output appeared roughly uniform and free of obvious short-range correlation. I then used the generated values in a Monte Carlo estimate of $\pi$, showing how pseudo-random numbers can support a real simulation-based calculation.

Compared with the original version, the Python notebook improves readability, modular structure, and visualization while preserving the core computational logic of the original project.